# Dự đoán VSL trực tiếp

Chạy trên máy có webcam bằng môi trường Python của dự án. Notebook tự chọn checkpoint `ensemble.pt` mới nhất trong `runs/`; có thể đặt `CHECKPOINT` trong ô cấu hình để chọn file khác.

| Phím | Tác dụng |
|---|---|
| `SPACE` | Thêm nhãn đã ổn định vào câu |
| `R` | Xóa câu và bắt đầu lại |
| `Q` hoặc `ESC` | Thoát |


In [1]:
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Hãy chạy Jupyter từ thư mục dự án hoặc notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.camera import open_camera
from src.model.predict import StreamConfig, VSLPredictor
from src.visualization.landmarks import draw

CHECKPOINT = None
CAMERA_INDEX = 0
CAMERA_RESOLUTION = None
MIRROR_DATA = False
MIRROR_PREVIEW = True
MIN_CONFIDENCE = 0.65
MIN_PROBABILITY_GAP = 0.08
DISPLAY_SIZE = 640
WINDOW_NAME = 'VSL realtime predict'

predictor = VSLPredictor(CHECKPOINT, min_confidence=MIN_CONFIDENCE, min_probability_gap=MIN_PROBABILITY_GAP)
stream_config = StreamConfig(mirror_data=MIRROR_DATA)
if predictor.feature_version < 2:
    print('Checkpoint cũ: hãy chạy reextract_features.ipynb rồi train.ipynb để dùng feature/model mới.')
print(f'Model: {predictor.checkpoint_path} | {len(predictor.classes)} lớp | input ({predictor.target_len}, {predictor.feature_dim}) | {predictor.device}')
print(f'Ngưỡng tin cậy: {predictor.threshold:.0%}')

Model: /Users/trungtran/Desktop/vsl-translator-new/runs/vsl_v5_20261119_204100/ensemble.pt | 16 lớp | input (30, 316) | mps
Ngưỡng tin cậy: 65%


In [2]:
FONT_CANDIDATES = [
    Path('/System/Library/Fonts/SFNS.ttf'),
    Path('/System/Library/Fonts/Supplemental/Arial Unicode.ttf'),
    Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'),
    Path('/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf'),
]
font_path = next((p for p in FONT_CANDIDATES if p.is_file()), None)
font = ImageFont.truetype(str(font_path), 25) if font_path else ImageFont.load_default()
small_font = ImageFont.truetype(str(font_path), 19) if font_path else ImageFont.load_default()

def fit_line(painter, text, text_font, width):
    if painter.textlength(text, font=text_font) <= width:
        return text
    while text and painter.textlength(text + '…', font=text_font) > width:
        text = text[:-1]
    return text + '…'

def render_preview(frame, pose_result, hand_result, result, sentence):
    preview = cv2.resize(frame, (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_AREA)
    if pose_result is not None:
        draw(preview, pose_result, point_color=(80, 190, 255), line_color=(110, 170, 200))
    if hand_result is not None:
        draw(preview, hand_result, point_color=(60, 230, 120), line_color=(200, 255, 220), secondary_color=(255, 190, 80))
    if MIRROR_PREVIEW:
        preview = cv2.flip(preview, 1)

    canvas = np.full((DISPLAY_SIZE + 170, DISPLAY_SIZE, 3), (35, 35, 35), dtype=np.uint8)
    canvas[:DISPLAY_SIZE] = preview
    image = Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    painter = ImageDraw.Draw(image)
    width = DISPLAY_SIZE - 32
    prediction = result['prediction']
    stable = result['stable_label']
    best_name = prediction['label'] if prediction else None
    confidence = prediction['confidence'] if prediction else 0.0
    title = f'{best_name}  {confidence:.0%}' if best_name else 'Chưa rõ'
    candidates = '  ·  '.join(f'{item["label"]} {item["confidence"]:.0%}' for item in prediction['top']) if prediction else ''
    status = {'idle': 'Chưa thấy tay', 'collecting': 'Đang ghi nhận', 'predicting': 'Đang dự đoán', 'ready': 'Sẵn sàng'}[result['status']]
    lines = [
        (f'Dự đoán: {title}', font, (110, 230, 160) if stable else (245, 245, 245)),
        (f'Ứng viên: {candidates or "—"}', small_font, (205, 205, 205)),
        (f'{status}  |  {result["duration"]:.1f}/{stream_config.window_seconds:.1f}s', small_font, (145, 205, 250)),
        (f'Câu: {" ".join(sentence[-6:]) or "—"}', font, (250, 220, 145)),
    ]
    for index, (text, text_font, color) in enumerate(lines):
        text_y = DISPLAY_SIZE + 8 + index * 40
        painter.text((16, text_y), fit_line(painter, text, text_font, width), font=text_font, fill=color)
    return cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2BGR)

## Chạy webcam

Bấm vào cửa sổ OpenCV để dùng phím điều khiển. `SPACE` chỉ thêm nhãn khi ba lần dự đoán gần nhất đồng ý.

In [ ]:
capture = open_camera(CAMERA_INDEX, CAMERA_RESOLUTION)

session = None
sentence = []

try:
    session = predictor.open_stream(stream_config)
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    while True:
        has_frame, raw_frame = capture.read()
        if not has_frame:
            print('Camera đã ngừng cung cấp hình ảnh.')
            break
        result = session.update_frame(raw_frame)
        preview = render_preview(session.last_frame, session.last_pose_result, session.last_hand_result, result, sentence)
        cv2.imshow(WINDOW_NAME, preview)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), ord('Q'), 27):
            break
        if key in (ord('r'), ord('R')):
            sentence.clear()
            session.reset()
        elif key == 32 and result['stable_label']:
            sentence.append(result['stable_label'])
            print('Câu:', ' '.join(sentence))
            session.reset()
finally:
    capture.release()
    if session is not None:
        session.close()
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)

print('Kết quả:', ' '.join(sentence) if sentence else '(trống)')

Camera 0: 1920x1080 @ 15 fps (mặc định 1920x1080, backend AVFOUNDATION)


I0000 00:00:1795177702.252994  155713 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795177702.332974  155722 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795177702.354266  155720 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795177702.373881  155741 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795177702.386930  155747 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795177702.578388  155715 landmark_projection_calculator.cc:81] Us

Kết quả: (trống)


: 